# Databricks Notebook: Master_Orchestrator.ipynb

# Este notebook orquestra a execução dos notebooks do pipeline ETL de países.

In [0]:
import json
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Define os caminhos base para os notebooks
# ATENÇÃO: Substitua <your-email-or-username> pelo seu email ou nome de usuário no Databricks
notebook_base_path = "/Users/<your-email-or-username>/pyspark-on-databricks/notebooks"

# Task 1: Extrair dados da API REST Countries
logger.info("Executando 01_extract_api_data.ipynb...")
raw_data_output_path = dbutils.notebook.run(
    f"{notebook_base_path}/01_extract_api_data",
    timeout_seconds=3600 # 1 hora de timeout
)
logger.info(f"01_extract_api_data.ipynb concluído. Saída: {raw_data_output_path}")

# Task 2: Processar dados com PySpark
logger.info("Executando 02_process_countries_data.ipynb...")
processed_countries_output_path = dbutils.notebook.run(
    f"{notebook_base_path}/02_process_countries_data",
    timeout_seconds=3600,
    arguments={"input_path": raw_data_output_path}
)
logger.info(f"02_process_countries_data.ipynb concluído. Saída: {processed_countries_output_path}")

# Task 3: Gerar dados econômicos simulados (pode rodar em paralelo)
logger.info("Executando 03_create_economic_data.ipynb...")
economic_data_output_path = dbutils.notebook.run(
    f"{notebook_base_path}/03_create_economic_data",
    timeout_seconds=3600
)
logger.info(f"03_create_economic_data.ipynb concluído. Saída: {economic_data_output_path}")

# Task 4: Enriquecer dados de países com dados econômicos
logger.info("Executando 04_enrich_countries_data.ipynb...")
enriched_countries_output_path = dbutils.notebook.run(
    f"{notebook_base_path}/04_enrich_countries_data",
    timeout_seconds=3600,
    arguments={
        "processed_countries_path": processed_countries_output_path,
        "economic_data_path": economic_data_output_path
    }
)
logger.info(f"04_enrich_countries_data.ipynb concluído. Saída: {enriched_countries_output_path}")

logger.info("Pipeline ETL de países concluído com sucesso!")
logger.info(f"Dados enriquecidos salvos em: {enriched_countries_output_path}")

# # Opcional: Retornar o caminho final para o Job do Databricks, se este notebook for executado como uma tarefa de Job
# dbutils.notebook.exit(enriched_countries_output_path)
